# 48 — Fail-closed gallery

Celebrate **correct rejection**: bad digests, invalid SC ranges, nonsense
parameters. Trust-building for safety and hardware readers.

## Honesty box

| | |
|---|---|
| **Proves** | Guards return null/raise/skip on malformed inputs as demonstrated. |
| **Does not prove** | Exhaustive security audit or formal proof of all APIs. |
| **Models** | `AdExNeuron` / `PerfectIntegratorNeuron` for happy-path contrast only. |


In [ ]:
from __future__ import annotations

import hashlib
import json
import re
from typing import Any

import numpy as np

from sc_neurocore import BitstreamEncoder
from sc_neurocore.neurons.models import AdExNeuron, PerfectIntegratorNeuron

HEX64 = re.compile(r"^[0-9a-f]{64}$")
print("SC-NeuroCore — NB-48 fail-closed gallery")


In [ ]:
def is_record(value: Any) -> bool:
    return isinstance(value, dict) or (
        hasattr(value, "__dict__") is False and isinstance(value, object)
        and not isinstance(value, (str, bytes, list, tuple, np.ndarray))
        and value is not None
        and not isinstance(value, (int, float, bool))
    )


def analysis_result_identity(result: Any) -> str | None:
    """Pedagogical mirror of FE evidenceCartIdentity (assertion-free)."""
    if not isinstance(result, dict):
        return None
    meta = result.get("analysis_metadata")
    if not isinstance(meta, dict):
        return None
    digest = meta.get("result_sha256")
    if not isinstance(digest, str):
        return None
    norm = digest.strip().lower()
    return norm if HEX64.match(norm) else None

cases = [
    ("ok digest", {"analysis_metadata": {"result_sha256": "a" * 64}}),
    ("uppercase ok", {"analysis_metadata": {"result_sha256": "B" * 64}}),
    ("short", {"analysis_metadata": {"result_sha256": "abc"}}),
    ("array root", [{"analysis_metadata": {"result_sha256": "a" * 64}}]),
    ("null meta", {"analysis_metadata": None}),
    ("meta list", {"analysis_metadata": [{"result_sha256": "a" * 64}]}),
]
for label, payload in cases:
    print(f"{label:16s} -> {analysis_result_identity(payload)}")


In [ ]:
# BitstreamEncoder: invalid range should fail closed
try:
    BitstreamEncoder(x_min=1.0, x_max=0.0, length=64, seed=0)
    print("NOTE (gap): BitstreamEncoder currently accepts inverted range — document, do not overclaim fail-closed here")
except Exception as exc:
    print("BitstreamEncoder inverted range rejected:", type(exc).__name__, str(exc)[:120])

try:
    enc = BitstreamEncoder(0.0, 1.0, length=0, seed=0)
    print("NOTE (gap): length=0 accepted:", enc)
except Exception as exc:
    print("BitstreamEncoder length=0 rejected:", type(exc).__name__, str(exc)[:120])

# Happy path HF neurons still run
v, s = AdExNeuron().simulate(500, current=200.0)
print(f"AdEx happy path spikes={s} len={len(v)}")
v2, s2 = PerfectIntegratorNeuron().simulate(200, current=2.0)
print(f"PerfectIntegrator happy path spikes={s2}")
print("NB-48 complete: rejections shown without crashing the kernel.")

